# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

In [1]:
import pandas as pd


In [2]:
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())

1 1.3333333333333333


In [3]:
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())

2


In [4]:
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))

3


In [5]:
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())

1005


In [6]:
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())

[9, 10]


Ваши ответы и объяснения:

1. 1 1.33 | s.isna().sum() находит 1 пропуск (None), а s.fillna(0) заменяет его на 0, поэтому среднее считается от трёх чисел [1, 0, 3]: $(1 + 0 + 3) / 3 = 1.333
2. 2 | .str.strip() убирает пробелы, .str.lower() превращает "A" в "a", остаются элементы ["a", "a", "b"] — уникальных среди них 2
3. 3 | f.drop_duplicates() вызывается без присваивания (df = df.drop_duplicates(),в Pandas методы удаления дублей возвращают новую таблицу, но не меняют исходную df на месте, поэтому в df остаются все 3 строки
4. 1005 | .str.replace(" ", "") убирает пробел внутри "1 000", pd.to_numeric() превращает строки в числа 1000 и 5, а .sum() складывает их в 1005
5. [9, 10] | pd.to_datetime() переводит строки в формат даты, а .dt.month достаёт номера месяцев

In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [18]:
# 1
def clean(path):
    df = pd.read_csv(path)
    print("После чтения:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("Суммы в числа:", len(df))

    CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("Города через словарь:", len(df))

    df = df.drop_duplicates().copy()
    print("Без дублей:", len(df))  
    
    df["order_date"] = pd.to_datetime(df["order_date"])
    print("Даты:", len(df))

    df = df[df["customer_id"] != "TEST"]
    print("Без TEST:", len(df))

    print("Проверки:", df["amount"].isna().sum(), df["city"].isna().sum(), df["order_id"].is_unique)

    return df

df_clean = clean("orders.csv")
    

После чтения: 5463
Суммы в числа: 5463
Города через словарь: 5463
Без дублей: 5283
Даты: 5283
Без TEST: 5280
Проверки: 0 0 True


После очистки осталось 5 280 реальных заказов, все дубли и тестовые записи удалены

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [22]:
# 2
sep = df_clean["order_date"] < "2026-10"  
octo = ~sep
paid = df_clean["status"] == "оплачен"
total_sep = df_clean.loc[sep & paid, "amount"].sum()
total_oct = df_clean.loc[octo & paid, "amount"].sum()
changes_total = (total_oct - total_sep)/total_sep * 100
print(total_sep)
print(total_oct)
print(f"Изменения октября к сентябрю: {changes_total:.2f}%")


35243410.0
34419520.0
Изменения октября к сентябрю: -2.34%


In [24]:
for city in sorted(df_clean["city"].unique()):
    in_city = df_clean["city"] == city
    city_sep = df_clean.loc[sep & paid & in_city, "amount"].sum()
    city_oct = df_clean.loc[octo & paid & in_city, "amount"].sum()
    city_change = (city_oct - city_sep)/city_sep * 100
    print(f"{city}: сентябрь = {city_sep:.0f}, октябрь = {city_oct:.0f}, изменение = {city_change:.2f}%")
    

Актобе: сентябрь = 2576960, октябрь = 2540420, изменение = -1.42%
Алматы: сентябрь = 14501070, октябрь = 14643020, изменение = 0.98%
Астана: сентябрь = 9034220, октябрь = 8850530, изменение = -2.03%
Атырау: сентябрь = 923990, октябрь = 969860, изменение = 4.96%
Караганда: сентябрь = 3648520, октябрь = 3371680, изменение = -7.59%
Шымкент: сентябрь = 4558650, октябрь = 4044010, изменение = -11.29%


Цифры полностью совпадают с чистыми данными из урока 4. Сильнее всех падает Шымкент (−11,29%), и именно он тянет вниз общую выручку всей компании (−2,34%).

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [25]:
# 3
df_raw = pd.read_csv("orders.csv")
df_raw["amount"] = pd.to_numeric(df_raw["amount"].str.replace(" ", ""))

df_no_dup = df_raw.drop_duplicates().copy()

sep_nd = df_no_dup["order_date"] < "2026-10"

oct_nd = ~sep_nd

paid_nd = df_no_dup["status"] == "оплачен"

s_nd = df_no_dup.loc[sep_nd & paid_nd, "amount"].sum()

o_nd = df_no_dup.loc[oct_nd & paid_nd, "amount"].sum()

print(f"После удаления дублей: {(o_nd - s_nd) / s_nd * 100:.1f}%")

После удаления дублей: -19.6%


| Шаг очистки | Убрано строк | Осталось строк | Изменение выручки (окт/сент) |
| :--- | :---: | :---: | :---: |
| Исходные данные (оплаченные) | 0 | 5 463 | -22,7% |
| Удаление полных дублей | 180 | 5 283 | -19.6% |
| Удаление тестового клиента TEST | 3 | 5 280 | -2,3% |

Cильнее всего цифру менеджера исказили 3 тестовых заказа (`TEST`) на огромные суммы — именно после их удаления «падение» выручки сократилось с -19.6% до реальных −2,3%.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [27]:
# 4
print(df_clean["delivery_days"].isna().sum())

158


In [29]:
no_deliv = df_clean["delivery_days"].isna()
no_deliv

0       False
1       False
2       False
3       False
4       False
        ...  
5458    False
5459    False
5460     True
5461    False
5462    False
Name: delivery_days, Length: 5280, dtype: bool

In [30]:
df_clean.loc[no_deliv, "city"].value_counts()

city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

In [31]:
df_clean.loc[no_deliv, "status"].value_counts()

status
оплачен    144
отменён     14
Name: count, dtype: int64

In [35]:
d = df_clean.loc[df_clean["city"] == "Шымкент", "delivery_days"]

In [38]:
print((d.dropna() <= 2).mean())

0.2984084880636605


In [39]:
print((d.fillna(d.median()) <= 2).mean())

0.28994845360824745


In [40]:
print((d.fillna(0) <= 2).mean())

0.31829896907216493


Честный вариант — dropna (29,8%), так как учитывает только заказы с известным сроком доставки; fillna(медиана) искусственно раздувает знаменатель и занижает долю до 29,0%, а fillna(0) превращает пропуски в «доставку за 0 дней» (0 <= 2 даёт True), завышая показатель до 31,8%.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [44]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

In [46]:
# 5
df = pd.read_csv("orders.csv")
df = df.drop_duplicates().copy()
df["city"] = df["city"].str.lower().str.strip().map(CITY)
print(df["city"].nunique(), len(df))

6 5283


1. Не было присваивания df = df.drop_duplicates().copy(), поэтому дубликаты не удалились. 2.Не были убраны пробелы (.str.strip()) и не приведены к одному языку написания городов (.map(CITY)).

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [49]:
days = df_clean["order_date"].dt.day_name().value_counts()
print(days)

order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64


In [50]:
print((days.min() - days.max()) / days.max() * 100)

-12.977099236641221


В понедельник заказов меньше всего 684 заказов. Cамый загруженный четверг 786 заказов. Разница между самым меньшим и самым загруженным -12.97%. В понедельник на смену нужно выпускать меньше курьеров. Влияет на смены курьеров.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8